
# 02_SQL_in_R

1.Install and load packages

In [ ]:
# Run this first - installs what we need
install.packages("sqldf")
install.packages("ggplot2")
install.packages("dplyr")

library(sqldf)
library(ggplot2)
library(dplyr)

cat("All packages loaded successfully\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




All packages loaded successfully


2.Load the clean data into R

In [ ]:
unzip('clean_data.zip', exdir = 'clean')

customers  <- read.csv('clean/customers.csv')
orders     <- read.csv('clean/orders.csv')
deliveries <- read.csv('clean/deliveries.csv')
drivers    <- read.csv('clean/drivers.csv')
vehicles   <- read.csv('clean/vehicles.csv')
hubs       <- read.csv('clean/hubs.csv')
incidents  <- read.csv('clean/incidents.csv')
complaints <- read.csv('clean/complaints.csv')
app_events <- read.csv('clean/app_events.csv')

cat("Tables loaded:\n")
cat("Orders:", nrow(orders), "rows\n")
cat("Deliveries:", nrow(deliveries), "rows\n")
cat("Customers:", nrow(customers), "rows\n")
cat("Drivers:", nrow(drivers), "rows\n")

Tables loaded:
Orders: 1250 rows
Deliveries: 950 rows
Customers: 650 rows
Drivers: 170 rows


### SQL Query 1: Delivery failure rate by zone

In [ ]:
library(sqldf)

# Which zones have the worst delivery performance?
query1 <- sqldf("
  SELECT
    o.pickup_zone,
    COUNT(*) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) AS on_time,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_rate_pct DESC
")

print(query1)

  pickup_zone total_deliveries failed delayed on_time failure_rate_pct avg_cost
1     Central              174     33      51      90            18.97    12.12
2       North              135     22      21      92            16.30    12.07
3   Riverside              119     18      25      76            15.13    12.39
4        West              114     14      21      79            12.28    11.94
5        East              156     19      31     106            12.18    12.57
6     Airport              113     12      31      70            10.62    17.08
7       South              139     14      22     103            10.07    12.48


### SQL Query 2: Hub performance ranking

In [ ]:
# Which hubs have the worst failure and delay rates?
query2 <- sqldf("
  SELECT
    h.hub_name,
    h.zone,
    h.hub_type,
    h.capacity_score,
    COUNT(*) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS delay_rate_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_name, h.zone, h.hub_type, h.capacity_score
  ORDER BY failure_rate_pct DESC
")

print(query2)

        hub_name      zone  hub_type capacity_score total_deliveries failed
1  Midtown Relay   Central  Charging             63              128     26
2   Central Core   Central   Control             88              115     23
3    Airport Hub   Airport  Dispatch             71              104     15
4      West Gate      West  Dispatch             69              127     16
5 North Exchange     North  Dispatch             82              136     17
6  Riverside Hub Riverside Warehouse             66              115     14
7     South Link     South  Dispatch             78              106     10
8      East Dock      East Warehouse             74              119     11
  delayed failure_rate_pct delay_rate_pct avg_cost avg_overrides
1      22            20.31          17.19    11.71          1.11
2      25            20.00          21.74    13.69          0.95
3      27            14.42          25.96    13.32          0.91
4      28            12.60          22.05    13.17      

### SQL Query 3: Driver performance analysis

In [ ]:
# Which drivers have the worst performance profiles?
query3 <- sqldf("
  SELECT
    dr.driver_id,
    dr.base_zone,
    dr.employment_type,
    dr.years_experience,
    dr.training_score,
    dr.driver_rating,
    COUNT(*) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY dr.driver_id, dr.base_zone, dr.employment_type,
           dr.years_experience, dr.training_score, dr.driver_rating
  ORDER BY failure_rate_pct DESC
  LIMIT 15
")

print(query3)

   driver_id base_zone employment_type years_experience training_score
1       D051      West        FullTime                3           75.4
2       D063     North        PartTime               12           85.7
3       D092      East        FullTime               15           88.2
4       D104      West        FullTime               15           87.7
5       D024 Riverside        PartTime                8           71.4
6       D103   Central        FullTime               15           72.5
7       D111   Airport        FullTime               15           79.2
8       D132     South        Contract                8           77.6
9       D147      West        FullTime               13           66.4
10      D170      West        FullTime               14           75.2
11      D010      West        FullTime                8           70.0
12      D005     North        FullTime                3           69.7
13      D095      West        FullTime               12           99.0
14    

### SQL Query 4: Service type cost and failure analysis

In [ ]:
# Which service types are most costly and failure-prone?
query4 <- sqldf("
  SELECT
    o.service_type,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_fuel_cost,
    ROUND(AVG(o.order_value), 2) AS avg_order_value,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.service_type
  ORDER BY failure_rate_pct DESC
")

print(query4)

  service_type total_orders failed delayed failure_rate_pct avg_fuel_cost
1     Business          126     25      28            19.84         13.14
2      Medical          108     16      22            14.81         12.77
3    Passenger          262     38      53            14.50         12.40
4       Retail          224     28      50            12.50         12.97
5       Parcel          230     25      49            10.87         13.08
  avg_order_value avg_overrides avg_rating
1           97.45          1.17       3.85
2           86.53          0.84       3.84
3           97.19          0.87       3.85
4           86.81          0.94       3.88
5           90.15          1.06       3.90


### SQL Query 5: Vehicle maintenance impact on deliveries

In [ ]:
# Do vehicles in poor condition cause more failures?
query5 <- sqldf("
  SELECT
    v.maintenance_status,
    v.vehicle_type,
    COUNT(*) AS total_deliveries,
    ROUND(AVG(v.battery_health_pct), 2) AS avg_battery_health,
    ROUND(AVG(v.odometer_km), 2) AS avg_odometer_km,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
  FROM deliveries d
  JOIN vehicles v ON d.vehicle_id = v.vehicle_id
  GROUP BY v.maintenance_status, v.vehicle_type
  ORDER BY failure_rate_pct DESC
")

print(query5)

   maintenance_status vehicle_type total_deliveries avg_battery_health
1            InRepair     CargoVan               68              69.82
2            InRepair       Diesel               55              75.65
3            InRepair       Hybrid               71              83.27
4            InRepair           EV               60              77.79
5           Scheduled     CargoVan               38              68.96
6              Active       Hybrid              131              74.30
7           Scheduled       Diesel                9              81.33
8              Active       Diesel               80              67.02
9              Active     CargoVan              117              74.41
10             Active           EV              214              82.86
11          Scheduled       Hybrid               42              80.56
12          Scheduled           EV               65              82.92
   avg_odometer_km failed delayed failure_rate_pct avg_cost
1        104131.0

### SQL Query 6: Customer complaints linked to delivery failures

In [ ]:
# Which customers have the most complaints linked to failed deliveries?
query6 <- sqldf("
  SELECT
    c.customer_id,
    c.home_zone,
    c.customer_type,
    c.loyalty_score,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT cp.complaint_id) AS total_complaints,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(AVG(cp.resolution_days), 2) AS avg_resolution_days,
    ROUND(SUM(cp.compensation_amount), 2) AS total_compensation
  FROM customers c
  JOIN orders o ON c.customer_id = o.customer_id
  JOIN deliveries d ON o.order_id = d.order_id
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY c.customer_id, c.home_zone, c.customer_type, c.loyalty_score
  HAVING total_complaints > 0
  ORDER BY total_complaints DESC
  LIMIT 15
")

print(query6)

   customer_id home_zone customer_type loyalty_score total_orders
1        C0368     North      Consumer          49.5            3
2        C0242      East      Consumer          83.8            3
3        C0282 Riverside      Consumer          71.4            2
4        C0421   Central      Consumer          59.0            3
5        C0001     North           SME          44.9            2
6        C0012   Airport      Consumer          46.3            3
7        C0015     South           SME          54.5            2
8        C0023     South      Consumer          73.1            5
9        C0054      West      Consumer          42.4            3
10       C0056     North      Consumer          71.1            2
11       C0078      East      Consumer          35.2            3
12       C0100   Central      Consumer          28.0            1
13       C0119   Central           SME          46.2            2
14       C0138     South      Consumer          93.2            1
15       C

### SQL Query 7: Incident types linked to delivery outcomes

In [ ]:
# What incident types most frequently lead to failed deliveries?
query7 <- sqldf("
  SELECT
    i.incident_type,
    i.severity,
    COUNT(*) AS total_incidents,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS linked_failures,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS linked_delays,
    ROUND(AVG(i.resolved_hours), 2) AS avg_resolution_hours,
    SUM(CASE WHEN i.resolution_status = 'Open' THEN 1 ELSE 0 END) AS still_open
  FROM incidents i
  JOIN deliveries d ON i.delivery_id = d.delivery_id
  GROUP BY i.incident_type, i.severity
  ORDER BY linked_failures DESC
")

print(query7)

      incident_type severity total_incidents linked_failures linked_delays
1      BatteryAlert   Medium              18               4             4
2      ProofMissing     High              12               4             3
3    CustomerNoShow     High              11               3             2
4      AppSyncError     High               6               2             2
5      AppSyncError   Medium              15               2             7
6      ProofMissing      Low              13               2             2
7    RouteDeviation      Low              17               2             2
8    SafetyNearMiss     High               4               2             1
9  TemperatureIssue Critical               5               2             0
10 TemperatureIssue     High               5               2             3
11 TemperatureIssue      Low               6               2             0
12     VehicleFault   Medium              16               2             2
13     AppSyncError      

## Optimised query with index simulation and EXPLAIN

In [ ]:
start_time <- proc.time()
query_base <- sqldf("
  SELECT
    o.pickup_zone,
    d.delivery_status,
    COUNT(*) AS total,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone, d.delivery_status
  ORDER BY o.pickup_zone
")
time_base <- proc.time() - start_time

# With index - optimised
start_time2 <- proc.time()
query_optimised <- sqldf(c(
  "CREATE INDEX idx_order_id ON deliveries(order_id)",
  "SELECT
    o.pickup_zone,
    d.delivery_status,
    COUNT(*) AS total,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone, d.delivery_status
  ORDER BY o.pickup_zone"
))
time_optimised <- proc.time() - start_time2

cat("=== Query Optimisation Results ===\n")
cat("Without index - elapsed time:", time_base["elapsed"], "seconds\n")
cat("With index    - elapsed time:", time_optimised["elapsed"], "seconds\n")
cat("\nQuery results:\n")
print(query_optimised)

Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


=== Query Optimisation Results ===
Without index - elapsed time: 0.098 seconds
With index    - elapsed time: 0.104 seconds

Query results:
   pickup_zone delivery_status total avg_cost
1      Airport         Delayed    31    18.28
2      Airport          Failed    12    17.22
3      Airport          OnTime    70    16.52
4      Central         Delayed    51    12.13
5      Central          Failed    33    13.44
6      Central          OnTime    90    11.64
7         East         Delayed    31    10.91
8         East          Failed    19    12.30
9         East          OnTime   106    13.10
10       North         Delayed    21    10.98
11       North          Failed    22    13.18
12       North          OnTime    92    12.06
13   Riverside         Delayed    25    11.88
14   Riverside          Failed    18    13.87
15   Riverside          OnTime    76    12.21
16       South         Delayed    22    14.35
17       South          Failed    14    11.98
18       South          OnTime   